# Stage 1 — Profiling the raw 311 data

**Read-only.** This notebook measures the data's problems so we can decide how to fix them in later stages. Nothing here changes any data.

It reads the Parquet copies in `data/interim/raw_parquet/` (made by `pipeline/convert_raw.py`), where every column is still text, exactly as in the CSVs.

We use **DuckDB**, which runs SQL directly on files without loading them into memory.

In [1]:
import duckdb
import pandas as pd
import pyogrio
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PARQUET = (ROOT / "data/interim/raw_parquet/*.parquet").as_posix()
GDB = (ROOT / "neighborhood_gdb/_ags_data66ACD992124244559E776E60316275B6.gdb").as_posix()

con = duckdb.connect()
# One view over all 17 files. union_by_name lines columns up by NAME, so the
# three different column layouts combine (missing columns become NULL).
# `yr` = the year taken from the file name.
con.sql(rf'''
    CREATE VIEW r AS
    SELECT *, regexp_extract(source_file, '(\d{{4}})', 1)::INT AS yr
    FROM read_parquet('{PARQUET}', union_by_name = true)
''')
q = lambda sql: con.sql(sql).df()

## 1. Column layouts

Each file has one of three layouts. This shows which columns each year has.

In [2]:
cols = {}
for f in sorted((ROOT / "data/interim/raw_parquet").glob("*.parquet")):
    yr = f.stem[-4:]
    cols[yr] = [c[0].lower() for c in con.sql(f"DESCRIBE SELECT * FROM '{f.as_posix()}'").fetchall()]
all_cols = sorted({c for v in cols.values() for c in v})
layout = pd.DataFrame({yr: ["✓" if c in v else "" for c in all_cols] for yr, v in cols.items()}, index=all_cols)
layout

,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
address,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
agency,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
closedate,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
councildistrict,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
createddate,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
duedate,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
esri_oid,,,,,,,,,,,,,,,,✓,
geolocation,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
hashedrecord,,,,,,,,,,,,,,,,✓,✓
isdeleted,,,,,,,,,,,,,,,,✓,✓


**Notes:**
- 2010–2024 share one layout. 2025 adds `NeedsSync`, `IsDeleted`, `HashedRecord` and `ESRI_OID`, and drops `RowID`. 2026 has `RowID` again and no `ESRI_OID`.
- `X`/`Y` (2026) and `x`/`y` (other years) are the same columns. DuckDB ignores letter case, so they merge automatically (names are shown in lowercase above).
- `x`/`y` repeat `Longitude`/`Latitude`, and `GeoLocation` repeats both as text.

## 2. Rows per year, and missing locations

In [3]:
summary = q('''
SELECT yr,
       count(*)                                                                  AS total_rows,
       count(*) FILTER (WHERE nullif(trim(Latitude),'') IS NOT NULL)              AS has_coords,
       count(*) FILTER (WHERE nullif(trim(Latitude),'') IS NULL
                          AND nullif(trim(Address),'')  IS NOT NULL)              AS no_coords_has_address,
       count(*) FILTER (WHERE nullif(trim(Latitude),'') IS NULL
                          AND nullif(trim(Address),'')  IS NULL)                  AS no_coords_no_address
FROM r GROUP BY yr ORDER BY yr
''')
summary.loc["TOTAL"] = summary.sum(numeric_only=True)
summary.loc["TOTAL", "yr"] = None
summary["pct_no_location"] = (100 * (summary.no_coords_has_address + summary.no_coords_no_address) / summary.total_rows).round(1)
summary

,yr,total_rows,has_coords,no_coords_has_address,no_coords_no_address,pct_no_location
0,2010.0,469534,469461,72,1,0.0
1,2011.0,440116,440102,14,0,0.0
2,2012.0,503449,503437,11,1,0.0
3,2013.0,533792,533787,5,0,0.0
4,2014.0,609569,557607,34867,17095,8.5
5,2015.0,669843,634678,17200,17965,5.2
6,2016.0,698099,666468,15290,16341,4.5
7,2017.0,671777,640632,15629,15516,4.6
8,2018.0,792440,745047,14140,33253,6.0
9,2019.0,767903,715464,1,52438,6.8


**What this shows:**
- About **13.0 million** rows in total.
- **No coordinates, but an address (~97k rows):** step 3 (address matching) can fill these. Almost all of them are from **2010–2018**.
- **No coordinates and no address (~2.2 million rows):** step 4 will drop these. From 2020 on, about 30% of each year looks like this. The next query shows what these requests are.

### What are the rows with no location at all?

In [4]:
q(r'''
SELECT SRType, count(*) AS n,
       round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
FROM r
WHERE nullif(trim(Latitude),'') IS NULL AND nullif(trim(Address),'') IS NULL
GROUP BY SRType ORDER BY n DESC LIMIT 15
''')

,SRType,n,pct
0,ECC-Information Request,2011858,90.8
1,ECC-DHCD Permit Customer Support Information,52807,2.4
2,BGE-StLight(s) Out,36128,1.6
3,FCPF-Trial Request-Parking,31959,1.4
4,ECC-Language Line,11750,0.5
5,FCPF-Court Petitions,11341,0.5
6,ECC-DHCD Permit Customer Support Escalation,7083,0.3
7,FCPF-Payments,5692,0.3
8,FCCS-Metered Water,5529,0.2
9,FCPF-Other,5515,0.2


About **91% are `ECC-Information Request`**: calls asking for information, which have no location by nature. Most of the rest are also not place-based (court petitions, payments, language line). Dropping them leaves the place-based requests. We should remember that **counts of all 311 calls** from this table will leave these calls out.

### Do rows without Latitude have the location anywhere else?

In [5]:
q(r'''
SELECT count(*) AS rows_without_lat,
       count(nullif(trim(x),''))            AS have_x,
       count(nullif(trim(GeoLocation),''))  AS have_geolocation,
       count(nullif(trim(ZipCode),''))      AS have_zip,
       count(nullif(trim(Neighborhood),'')) AS have_neighborhood
FROM r WHERE nullif(trim(Latitude),'') IS NULL
''')

,rows_without_lat,have_x,have_geolocation,have_zip,have_neighborhood
0,2312152,0,0,55740,44


No. When `Latitude` is missing, `x` and `GeoLocation` are missing too, so there's no other column to recover the location from.

## 3. Request IDs: which column can be the primary key?

A primary key must be **unique** and **never empty**.

In [6]:
q(r'''
SELECT count(*)                          AS total_rows,
       count(DISTINCT ServiceRequestNum) AS distinct_ServiceRequestNum,
       count(DISTINCT SRRecordID)        AS distinct_SRRecordID,
       count(*) FILTER (WHERE ServiceRequestNum IS NULL) AS null_ServiceRequestNum,
       count(*) FILTER (WHERE SRRecordID IS NULL)        AS null_SRRecordID
FROM r
''')

,total_rows,distinct_ServiceRequestNum,distinct_SRRecordID,null_ServiceRequestNum,null_SRRecordID
0,13019163,13019163,13019163,0,0


**Both** columns are unique across all 13 million rows, and neither is ever empty, so the yearly files don't overlap (no duplicates).
- `ServiceRequestNum` (e.g. `26-00004250`) is human-readable and has one consistent format.
- `SRRecordID` changed format over time (`1032622290` in 2010, `500VO00000ta00DYAQ` in 2026).

The ArcGIS API (stage 8) returns both. Upserts need a key that stays the same over time.

In [7]:
q(r'''
SELECT yr, any_value(ServiceRequestNum) AS example_srn, any_value(SRRecordID) AS example_srid,
       min(length(ServiceRequestNum)) AS srn_len_min, max(length(ServiceRequestNum)) AS srn_len_max
FROM r GROUP BY yr ORDER BY yr
''')

,yr,example_srn,example_srid,srn_len_min,srn_len_max
0,2010,12-00778913,1032622290,11,11
1,2011,11-00690934,1007027615,11,11
2,2012,12-00723209,1031286594,11,11
3,2013,13-00697499,1055368260,11,11
4,2014,14-00099627,5004100000dAfM7AAK,11,11
5,2015,18-00919289,5001K00000jmGMAQA2,11,11
6,2016,16-00769297,5004100000dKR4oAAG,11,11
7,2017,17-00454887,5004100000dHD4pAAG,11,11
8,2018,18-00365488,5004100000dLtPEAA0,11,11
9,2019,19-00830663,5004M00000afabsQAA,11,11


## 4. `IsDeleted` and `NeedsSync` (2025–2026 only)

In [8]:
q("SELECT yr, IsDeleted, NeedsSync, count(*) AS n FROM r WHERE yr >= 2025 GROUP BY ALL ORDER BY ALL")

,yr,IsDeleted,NeedsSync,n
0,2025,0,1,931324
1,2026,0,1,975035


`IsDeleted` is **0 on every row**, so there are no deleted rows to handle. `NeedsSync` is always 1, an internal flag of the source system that carries no information.

## 5. Address formats: why matching needs cleanup

The same street address is written in several different ways:

In [9]:
q(r'''
SELECT CASE
         WHEN Address ILIKE '%, Baltimore City, %' THEN 'street, Baltimore City, zip'
         WHEN Address ILIKE '%, BC, %'             THEN 'street, BC, zip'
         WHEN Address ILIKE '%, BALTIMORE, MD%'    THEN 'street, BALTIMORE, MD[, zip]'
         WHEN Address ILIKE '%, BALTIMORE COUNTY%' THEN 'street, BALTIMORE COUNTY, zip'
         WHEN regexp_matches(Address, ',\s*\d{5}$') THEN 'street, zip'
         WHEN Address NOT LIKE '%,%'               THEN 'street only'
         ELSE 'other (e.g. street, TOWN, zip)'
       END AS format,
       count(*) AS n, any_value(Address) AS example
FROM r WHERE nullif(trim(Address),'') IS NOT NULL
GROUP BY 1 ORDER BY n DESC
''')

,format,n,example
0,"street, Baltimore City, zip",9373542,"516 S WASHINGTON ST, Baltimore City, 21231"
1,"street, BALTIMORE, MD[, zip]",550135,"1131 LIGHT ST, BALTIMORE, MD, 21230"
2,"other (e.g. street, TOWN, zip)",235780,"3300 E BALTIMORE ST, Baltimore City"
3,"street, zip",230253,"29 RAMSGATE CT, 21227"
4,street only,196078,HUNTINGDON AVE
5,"street, BC, zip",164103,"7206 CHALKSTONE DR, BC, 21208"
6,"street, BALTIMORE COUNTY, zip",54269,"2500 FARRINGDON RD, BALTIMORE COUNTY, 21209"


There are also double spaces (`3414  PARK-HEIGHTS AV`), abbreviation differences (`AV` vs `AVE`), mixed upper/lower case, and intersections (`A ST & B AVE`).

**How much the cleanup matters.** Here is the number of rows step 3 could fill under each matching rule:

In [10]:
fill = lambda key_sql: q(f'''
WITH n AS (SELECT *, {key_sql} AS k FROM r),
known AS (SELECT DISTINCT k FROM n WHERE nullif(trim(Latitude),'') IS NOT NULL AND k <> '')
SELECT count(*) AS need_fill, count(known.k) AS matchable
FROM n LEFT JOIN known USING (k)
WHERE nullif(trim(Latitude),'') IS NULL AND n.k <> ''
''')
rules = {
    "A. exact (uppercase + trim)":            "upper(trim(Address))",
    "B. street part only (before 1st comma)": r"upper(regexp_replace(trim(split_part(Address, ',', 1)), '\s+', ' ', 'g'))",
}
pd.concat({name: fill(sql) for name, sql in rules.items()}).droplevel(1).assign(
    pct=lambda d: (100 * d.matchable / d.need_fill).round(1))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,need_fill,matchable,pct
A. exact (uppercase + trim),97230,24676,25.4
B. street part only (before 1st comma),97119,75519,77.8


Keeping only the street part (before the first comma) and squeezing extra spaces raises fills from about 25% to about 78%. Standardizing words in Stage 3 (`AV`→`AVE`, `STREET`→`ST`, …) should raise it a bit more.

⚠️ **Watch out:** removing the town or zip can merge a **city** address with a **county** address on the same street name (for example a `MAIN ST` in both). Stage 3 needs to handle this.

## 6. Coordinates that look wrong

Rough box around Baltimore City: latitude 39.15–39.42, longitude −76.76 to −76.48. This box is only a quick first check. Stage 5 uses the real neighborhood boundaries.

In [11]:
q(r'''
SELECT count(*) AS rows_with_coords,
       count(*) FILTER (WHERE try_cast(Latitude AS DOUBLE) IS NULL) AS unparseable,
       count(*) FILTER (WHERE try_cast(Latitude AS DOUBLE) = 0)      AS zero,
       count(*) FILTER (WHERE NOT (try_cast(Latitude AS DOUBLE)  BETWEEN 39.15  AND 39.42
                               AND try_cast(Longitude AS DOUBLE) BETWEEN -76.76 AND -76.48)) AS outside_box
FROM r WHERE nullif(trim(Latitude),'') IS NOT NULL
''')

,rows_with_coords,unparseable,zero,outside_box
0,10707011,0,0,86007


In [12]:
q(r'''
SELECT yr, Latitude, Longitude, Address FROM r
WHERE NOT (try_cast(Latitude AS DOUBLE)  BETWEEN 39.15  AND 39.42
       AND try_cast(Longitude AS DOUBLE) BETWEEN -76.76 AND -76.48)
ORDER BY random() LIMIT 8
''')

,yr,Latitude,Longitude,Address
0,2026,39.363672463744784,-76.763127621876109,"3706 PARKFIELD RD, PIKESVILLE, 21208"
1,2018,39.345355521254300,-76.479616350965200,"8911 LENNINGS LN, ROSEDALE, MD, 21237"
2,2021,39.310556990585690,-76.781025188782220,"7899 CANTWELL RD, WINDSOR MILL, 21244"
3,2023,39.371210916817520,-76.795661107157840,"9016 SAMOSET RD, RANDALLSTOWN, 21133"
4,2015,39.229768000000000,-76.457108000000000,"2414 ESTELLE AVE, SPARROWS POINT, 21219"
5,2016,39.361792000000000,-76.460486000000000,"35 CATERHAM CT, ROSEDALE, 21237"
6,2023,39.287266884167074,-76.449445926960120,"1206 MARITIME CIR, ESSEX, 21221"
7,2014,39.400554000000000,-76.463825000000000,"9127 SNYDER LN, BALTIMORE COUNTY, 21128"


Every coordinate is a valid number and none is zero. About **86k** points fall outside the box, and they look **real**: they are **Baltimore County** addresses (Essex, Nottingham, Reisterstown), not errors. As you decided, they'll be kept with an empty `nsa_id`.

## 7. Date formats

In [13]:
q(r'''
SELECT yr, any_value(CreatedDate) AS created_example, any_value(CloseDate) AS close_example
FROM r WHERE yr IN (2010, 2018, 2024, 2025, 2026) GROUP BY yr ORDER BY yr
''')

,yr,created_example,close_example
0,2010,10/1/2010 3:56:00 AM,10/1/2010 11:08:00 AM
1,2018,1/22/2018 11:43:41 AM,2/6/2018 4:30:40 PM
2,2024,5/6/2024 4:09:32 PM,5/22/2024 3:19:03 PM
3,2025,1/4/2025 8:24:59 AM,1/8/2025 2:35:00 PM
4,2026,2026/03/16 12:31:03+00,2026/03/27 08:34:02+00


There are two formats:
- **2010–2025:** `3/24/2010 7:21:00 AM` (month/day/year, 12-hour clock, no time zone)
- **2026:** `2026/01/03 14:29:00+00` (year/month/day, 24-hour clock, **UTC**)

Stage 2 needs to check whether the older dates are local Baltimore time. If they are, they must be converted to UTC so everything lines up.

## 8. Neighborhood data: geodatabase vs the Downloads CSV

In [14]:
g = pyogrio.read_dataframe(GDB, read_geometry=False)
c = pd.read_csv(Path.home() / "Downloads/Neighborhood_Statistical_Area_(NSA)_Boundaries.csv", encoding="utf-8-sig")
print("GDB:", g.shape, "| CSV:", c.shape)
print("Columns only in CSV:", sorted(set(c.columns) - set(g.columns)))
print("Columns only in GDB:", sorted(set(g.columns) - set(c.columns)))
print("Same 279 neighborhood names:", set(g.Name) == set(c.Name), "| names unique:", g.Name.is_unique)
m = g.merge(c, on="Name", suffixes=("_g", "_c"))
num = [x for x in g.columns if x not in ("Name", "GlobalID")]
diffs = [x for x in num
         if not (pd.to_numeric(m[x + "_g"]).round(4).fillna(-1) == pd.to_numeric(m[x + "_c"]).round(4).fillna(-1)).all()]
print("Columns whose values differ:", diffs or "none")

GDB: (279, 60) | CSV: (279, 61)
Columns only in CSV: ['OBJECTID']
Columns only in GDB: []
Same 279 neighborhood names: True | names unique: True
Columns whose values differ: none


In [15]:
info = pyogrio.read_info(GDB)
print("Layer CRS:", info["crs"], "| geometry:", info["geometry_type"], "| features:", info["features"])
g[["Name", "Population", "Total_Units", "Occ_Vacant", "Med_Age", "GlobalID"]].head()

Layer CRS: EPSG:2248 | geometry: MultiPolygon | features: 279


,Name,Population,Total_Units,Occ_Vacant,Med_Age,GlobalID
0,Beechfield,4013.0,1721.0,129.0,37.400002,{37F14B9B-47AD-455C-B49F-09C75C1EBFD1}
1,Belair-Parkside,449.0,174.0,22.0,37.099998,{37831474-65E5-4D7B-8E7A-559B904B2A05}
2,Curtis Bay Industrial Area,20.0,1.0,0.0,9.500000,{C708A66B-72BA-495F-A3FA-722576272A66}
3,Eastwood,764.0,286.0,14.0,33.599998,{C74BB694-268C-4801-AFD2-3E38B488D732}
4,Forest Park Golf Course,358.0,135.0,12.0,45.900002,{8A32528F-ED10-49F9-95E8-4E9EB8F7034F}


The values are **identical**, and only the geodatabase has the boundary shapes, so **we'll use the geodatabase**.

For the `neighborhoods` primary key there are two options: `Name`, which is unique, or `GlobalID`, a stable ID from the source system.

The boundaries are in **EPSG:2248** (Maryland State Plane, feet). Stage 5 will convert them to latitude/longitude (EPSG:4326).